In [1]:
import os
import numpy as np
import pandas as pd

# find the mounted feature-extraction output
BASE = None
for root in os.listdir("/kaggle/input"):
    cand = f"/kaggle/input/{root}"
    for r, d, files in os.walk(cand):
        if any(f.endswith(".npy") for f in files):
            BASE = r
            break
    if BASE: break

print("Embeddings folder:", BASE)
print("\nFiles found:")
for f in sorted(os.listdir(BASE)):
    print("  ", f)

# quick load test
emb = np.load(f"{BASE}/resnet_baseline_ddi.npy")
meta = pd.read_csv(f"{BASE}/meta_ddi.csv")
print(f"\nSanity: resnet DDI embeddings {emb.shape}, meta rows {len(meta)}")
print("Meta columns:", meta.columns.tolist())

Embeddings folder: /kaggle/input/notebooks/nirajankunwor/feature-extraction/embeddings

Files found:
   dermlip_ddi.npy
   dermlip_hamisic_test.npy
   dermlip_scin.npy
   dinov3_ddi.npy
   dinov3_hamisic_test.npy
   dinov3_scin.npy
   meta_ddi.csv
   meta_hamisic_test.csv
   meta_scin.csv
   monet_ddi.npy
   monet_hamisic_test.npy
   monet_scin.npy
   resnet_baseline_ddi.npy
   resnet_baseline_hamisic_test.npy
   resnet_baseline_scin.npy

Sanity: resnet DDI embeddings (656, 2048), meta rows 656
Meta columns: ['image_path', 'label', 'malignant', 'skin_tone', 'patient_id', 'source_dataset']


In [2]:
import os, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit

BASE = "/kaggle/input/notebooks/nirajankunwor/feature-extraction/embeddings"
MODELS = ["resnet_baseline", "dermlip", "monet", "dinov3"]

# per-dataset column names (they differ!)
LABEL_COL = {"ddi": "label", "scin": "label", "hamisic_test": "mapped_label"}
GROUP_COL = {"ddi": "patient_id", "scin": "patient_id", "hamisic_test": "pid"}

def load(model, dataset):
    emb = np.load(f"{BASE}/{model}_{dataset}.npy")
    meta = pd.read_csv(f"{BASE}/meta_{dataset}.csv")
    assert len(emb) == len(meta), f"mismatch {model} {dataset}"
    return emb, meta

def patient_probe(emb, meta, label_col, group_col, min_class_count=10, seed=42):
    df = meta.copy()
    vc = df[label_col].value_counts()
    keep = vc[vc >= min_class_count].index
    mask = df[label_col].isin(keep).values
    emb_f, df_f = emb[mask], df[mask].reset_index(drop=True)

    y = df_f[label_col].values
    groups = df_f[group_col].astype(str).values

    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
    tr, te = next(gss.split(emb_f, y, groups))

    scaler = StandardScaler().fit(emb_f[tr])
    Xtr, Xte = scaler.transform(emb_f[tr]), scaler.transform(emb_f[te])

    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(Xtr, y[tr])
    pred = clf.predict(Xte)
    bacc = balanced_accuracy_score(y[te], pred)
    return bacc, te, y[te], pred, df_f.iloc[te], len(keep), len(df_f)

# ============================================================
# In-domain reference (HAM/ISIC test) + Distribution effect (SCIN)
# ============================================================
print(f"{'Model':<16} {'In-domain':<12} {'SCIN':<12} {'Drop':<8}")
print("-"*50)
results = {}
for m in MODELS:
    emb_id, meta_id = load(m, "hamisic_test")
    bacc_id, *_ , nclass_id, nimg_id = patient_probe(
        emb_id, meta_id, LABEL_COL["hamisic_test"], GROUP_COL["hamisic_test"])

    emb_sc, meta_sc = load(m, "scin")
    bacc_sc, *_ , nclass_sc, nimg_sc = patient_probe(
        emb_sc, meta_sc, LABEL_COL["scin"], GROUP_COL["scin"])

    drop = bacc_id - bacc_sc
    results[m] = {"in_domain": bacc_id, "scin": bacc_sc, "drop": drop}
    print(f"{m:<16} {bacc_id:<12.3f} {bacc_sc:<12.3f} {drop:<8.3f}")

print(f"\nIn-domain: {nclass_id} classes, {nimg_id} images")
print(f"SCIN: {nclass_sc} classes, {nimg_sc} images (after min_class_count=10 filter)")

Model            In-domain    SCIN         Drop    
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


resnet_baseline  0.617        0.060        0.557   


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


dermlip          0.518        0.199        0.319   


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


monet            0.519        0.138        0.381   
dinov3           0.472        0.131        0.341   

In-domain: 8 classes, 5378 images
SCIN: 70 classes, 6033 images (after min_class_count=10 filter)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [3]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

BASE = "/kaggle/input/notebooks/nirajankunwor/feature-extraction/embeddings"
MODELS = ["resnet_baseline", "dermlip", "monet", "dinov3"]
TONE_MAP = {12: "FST I-II", 34: "FST III-IV", 56: "FST V-VI"}

def load(model, dataset):
    emb = np.load(f"{BASE}/{model}_{dataset}.npy")
    meta = pd.read_csv(f"{BASE}/meta_{dataset}.csv")
    return emb, meta

# ============================================================
# DDI TONE EFFECT: binary malignant classification, split by skin tone
# ============================================================
print(f"{'Model':<16} {'Overall':<9} {'FST I-II':<10} {'FST III-IV':<12} {'FST V-VI':<10} {'ToneGap':<8}")
print("-"*70)

tone_results = {}
for m in MODELS:
    emb, meta = load(m, "ddi")
    df = meta.copy()
    # binary target: malignant True/False -> 1/0
    y = df["malignant"].astype(int).values
    groups = df["patient_id"].astype(str).values
    tone = df["skin_tone"].values

    # patient-level split
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    tr, te = next(gss.split(emb, y, groups))

    scaler = StandardScaler().fit(emb[tr])
    Xtr, Xte = scaler.transform(emb[tr]), scaler.transform(emb[te])

    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(Xtr, y[tr])
    pred = clf.predict(Xte)

    # overall balanced accuracy
    overall = balanced_accuracy_score(y[te], pred)

    # per-tone balanced accuracy (on the test set)
    tone_te = tone[te]
    y_te = y[te]
    per_tone = {}
    for code, name in TONE_MAP.items():
        mask = tone_te == code
        if mask.sum() > 0 and len(np.unique(y_te[mask])) > 1:
            per_tone[name] = balanced_accuracy_score(y_te[mask], pred[mask])
        elif mask.sum() > 0:
            # only one class present in this tone bin's test set
            per_tone[name] = np.mean(pred[mask] == y_te[mask])
        else:
            per_tone[name] = np.nan

    vals = [per_tone.get(TONE_MAP[c], np.nan) for c in [12,34,56]]
    tone_gap = np.nanmax(vals) - np.nanmin(vals)
    tone_results[m] = {"overall": overall, **per_tone, "gap": tone_gap}

    print(f"{m:<16} {overall:<9.3f} "
          f"{per_tone.get('FST I-II', np.nan):<10.3f} "
          f"{per_tone.get('FST III-IV', np.nan):<12.3f} "
          f"{per_tone.get('FST V-VI', np.nan):<10.3f} "
          f"{tone_gap:<8.3f}")

# quick check of tone distribution in the test split
print("\nTest-set tone distribution:")
print(pd.Series([TONE_MAP.get(t, t) for t in tone[te]]).value_counts())

Model            Overall   FST I-II   FST III-IV   FST V-VI   ToneGap 
----------------------------------------------------------------------
resnet_baseline  0.670     0.650      0.718        0.631      0.087   
dermlip          0.729     0.793      0.661        0.732      0.132   
monet            0.684     0.732      0.703        0.628      0.104   
dinov3           0.660     0.782      0.644        0.574      0.208   

Test-set tone distribution:
FST V-VI      77
FST III-IV    65
FST I-II      55
Name: count, dtype: int64


In [4]:
import warnings
warnings.filterwarnings("ignore")  # silence the 70-class metric warnings

import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit

BASE = "/kaggle/input/notebooks/nirajankunwor/feature-extraction/embeddings"
MODELS = ["resnet_baseline", "dermlip", "monet", "dinov3"]
TONE_MAP = {12: "FST I-II", 34: "FST III-IV", 56: "FST V-VI"}
N_BOOT = 1000
rng = np.random.default_rng(42)

def load(model, dataset):
    return np.load(f"{BASE}/{model}_{dataset}.npy"), pd.read_csv(f"{BASE}/meta_{dataset}.csv")

def fit_probe(emb, y, groups, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
    tr, te = next(gss.split(emb, y, groups))
    scaler = StandardScaler().fit(emb[tr])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(scaler.transform(emb[tr]), y[tr])
    return te, y[te], clf.predict(scaler.transform(emb[te]))

def boot_ci(y_true, y_pred, metric_fn, n=N_BOOT):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    stats = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        try: stats.append(metric_fn(y_true[idx], y_pred[idx]))
        except Exception: pass
    return np.mean(stats), np.percentile(stats, 2.5), np.percentile(stats, 97.5)

# PART 1: DISTRIBUTION EFFECT (SCIN)
print("DISTRIBUTION EFFECT — SCIN balanced accuracy [95% CI]")
print("-"*55)
scin_results = {}
for m in MODELS:
    emb, meta = load(m, "scin")
    vc = meta["label"].value_counts(); keep = vc[vc >= 10].index
    mask = meta["label"].isin(keep).values
    y = meta[mask]["label"].values; groups = meta[mask]["patient_id"].astype(str).values
    te, y_te, pred = fit_probe(emb[mask], y, groups)
    mean, lo, hi = boot_ci(y_te, pred, balanced_accuracy_score)
    scin_results[m] = (mean, lo, hi)
    print(f"{m:<16} {mean:.3f} [{lo:.3f}, {hi:.3f}]")

# PART 2: TONE EFFECT (DDI)
print("\nTONE EFFECT — DDI malignant accuracy per tone [95% CI]")
print("-"*55)
tone_results = {}
for m in MODELS:
    emb, meta = load(m, "ddi")
    y = meta["malignant"].astype(int).values
    groups = meta["patient_id"].astype(str).values
    tone = meta["skin_tone"].values
    te, y_te, pred = fit_probe(emb, y, groups)
    tone_te = tone[te]
    print(f"\n{m}:")
    for code, name in TONE_MAP.items():
        mask = tone_te == code
        if mask.sum() < 3: continue
        mean, lo, hi = boot_ci(y_te[mask], pred[mask], lambda a,b:(a==b).mean())
        print(f"  {name:<12} n={mask.sum():<3} {mean:.3f} [{lo:.3f}, {hi:.3f}]")
    # bootstrap the gap
    gaps = []
    for _ in range(N_BOOT):
        accs = []
        for code in TONE_MAP:
            mask = tone_te == code
            if mask.sum() < 3: continue
            idx = rng.integers(0, mask.sum(), mask.sum())
            accs.append((y_te[mask][idx] == pred[mask][idx]).mean())
        if len(accs) >= 2: gaps.append(max(accs)-min(accs))
    g_m, g_lo, g_hi = np.mean(gaps), np.percentile(gaps,2.5), np.percentile(gaps,97.5)
    tone_results[m] = (g_m, g_lo, g_hi)
    print(f"  TONE GAP: {g_m:.3f} [{g_lo:.3f}, {g_hi:.3f}]")

# SUMMARY: the headline comparison
print("\n" + "="*55)
print("HEADLINE: distribution effect vs tone effect")
print("="*55)
for m in MODELS:
    sc = scin_results[m][0]
    tg = tone_results[m][0]
    print(f"{m:<16} SCIN_bacc={sc:.3f}  tone_gap={tg:.3f}")

DISTRIBUTION EFFECT — SCIN balanced accuracy [95% CI]
-------------------------------------------------------
resnet_baseline  0.062 [0.050, 0.077]
dermlip          0.199 [0.171, 0.230]
monet            0.136 [0.111, 0.163]
dinov3           0.129 [0.106, 0.152]

TONE EFFECT — DDI malignant accuracy per tone [95% CI]
-------------------------------------------------------

resnet_baseline:
  FST I-II     n=55  0.692 [0.564, 0.800]
  FST III-IV   n=65  0.816 [0.723, 0.908]
  FST V-VI     n=77  0.767 [0.675, 0.857]
  TONE GAP: 0.143 [0.031, 0.280]

dermlip:
  FST I-II     n=55  0.818 [0.709, 0.927]
  FST III-IV   n=65  0.707 [0.600, 0.815]
  FST V-VI     n=77  0.806 [0.714, 0.883]
  TONE GAP: 0.136 [0.029, 0.260]

monet:
  FST I-II     n=55  0.778 [0.673, 0.891]
  FST III-IV   n=65  0.771 [0.677, 0.862]
  FST V-VI     n=77  0.741 [0.649, 0.831]
  TONE GAP: 0.096 [0.018, 0.203]

dinov3:
  FST I-II     n=55  0.821 [0.709, 0.909]
  FST III-IV   n=65  0.707 [0.585, 0.816]
  FST V-VI     n=77 

In [5]:
import pandas as pd
BASE = "/kaggle/input/notebooks/nirajankunwor/feature-extraction/embeddings"
meta = pd.read_csv(f"{BASE}/meta_scin.csv")
vc = meta["label"].value_counts()
print(f"Total unique labels: {len(vc)}\n")
for label, count in vc.items():
    print(f"{count:5d}  {label}")

Total unique labels: 211

 1079  Eczema
  590  Allergic Contact Dermatitis
  442  Urticaria
  401  Insect Bite
  306  Folliculitis
  234  Psoriasis
  211  Tinea
  136  Impetigo
  130  Herpes Zoster
  129  Drug Rash
  128  Pigmented purpuric eruption
  123  Acne
  109  Herpes Simplex
   98  CD - Contact dermatitis
   93  Acute dermatitis, NOS
   93  Pityriasis rosea
   77  Keratosis pilaris
   72  Irritant Contact Dermatitis
   72  Tinea Versicolor
   69  Lichen Simplex Chronicus
   62  Lichen planus/lichenoid eruption
   59  Stasis Dermatitis
   57  Rosacea
   57  Leukocytoclastic Vasculitis
   56  Granuloma annulare
   54  Prurigo nodularis
   53  O/E - ecchymoses present
   53  Viral Exanthem
   51  Hypersensitivity
   46  Verruca vulgaris
   45  Abrasion, scrape, or scab
   44  Photodermatitis
   44  Scabies
   41  Cellulitis
   39  Acute and chronic dermatitis
   36  Intertrigo
   36  Scar Condition
   35  Perioral Dermatitis
   34  Purpura
   34  Abscess
   33  Molluscum Contagios

In [6]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit

BASE = "/kaggle/input/notebooks/nirajankunwor/feature-extraction/embeddings"
MODELS = ["resnet_baseline", "dermlip", "monet", "dinov3"]
rng = np.random.default_rng(42)

# ============================================================
# CLINICAL CATEGORY MAPPING (based on standard dermatology grouping)
# ============================================================
CATEGORY_MAP = {
    # --- INFLAMMATORY / ECZEMATOUS ---
    "Eczema": "inflammatory", "Allergic Contact Dermatitis": "inflammatory",
    "Urticaria": "inflammatory", "Psoriasis": "inflammatory",
    "Drug Rash": "inflammatory", "Acne": "inflammatory",
    "CD - Contact dermatitis": "inflammatory", "Acute dermatitis, NOS": "inflammatory",
    "Pityriasis rosea": "inflammatory", "Keratosis pilaris": "inflammatory",
    "Irritant Contact Dermatitis": "inflammatory", "Lichen Simplex Chronicus": "inflammatory",
    "Lichen planus/lichenoid eruption": "inflammatory", "Stasis Dermatitis": "inflammatory",
    "Rosacea": "inflammatory", "Prurigo nodularis": "inflammatory",
    "Hypersensitivity": "inflammatory", "Photodermatitis": "inflammatory",
    "Acute and chronic dermatitis": "inflammatory", "Intertrigo": "inflammatory",
    "Perioral Dermatitis": "inflammatory", "Seborrheic Dermatitis": "inflammatory",
    "Chronic dermatitis, NOS": "inflammatory", "Miliaria": "inflammatory",
    "Granuloma annulare": "inflammatory",
    # --- INFECTIOUS ---
    "Insect Bite": "infectious", "Folliculitis": "infectious", "Tinea": "infectious",
    "Impetigo": "infectious", "Herpes Zoster": "infectious", "Herpes Simplex": "infectious",
    "Tinea Versicolor": "infectious", "Viral Exanthem": "infectious",
    "Verruca vulgaris": "infectious", "Scabies": "infectious", "Cellulitis": "infectious",
    "Molluscum Contagiosum": "infectious", "Ecthyma": "infectious",
    "Onychomycosis": "infectious", "Infected eczema": "infectious",
    # --- VASCULAR / PURPURIC ---
    "Pigmented purpuric eruption": "vascular_purpuric", "Leukocytoclastic Vasculitis": "vascular_purpuric",
    "Purpura": "vascular_purpuric", "O/E - ecchymoses present": "vascular_purpuric",
    "Livedo reticularis": "vascular_purpuric",
    # --- NEOPLASTIC / PIGMENTED LESIONS ---
    "Actinic Keratosis": "neoplastic", "SCC/SCCIS": "neoplastic",
    "Hemangioma": "neoplastic", "Dermatofibroma": "neoplastic", "SK/ISK": "neoplastic",
    "Cutaneous T Cell Lymphoma": "neoplastic", "Melanocytic Nevus": "neoplastic",
    "Basal Cell Carcinoma": "neoplastic",
    # --- TRAUMATIC / OTHER (grouped) ---
    "Abrasion, scrape, or scab": "traumatic_other", "Scar Condition": "traumatic_other",
    "Abscess": "traumatic_other", "Inflicted skin lesions": "traumatic_other",
    "Xerosis": "traumatic_other", "Erythema multiforme": "inflammatory",
    "Post-Inflammatory hyperpigmentation": "pigmentary", "Erythema ab igne": "pigmentary",
    "Cutaneous lupus": "inflammatory", "Pityriasis lichenoides": "inflammatory",
    "Superficial wound of body region": "traumatic_other", "Lichen nitidus": "inflammatory",
    "Cutaneous sarcoidosis": "inflammatory", "Hidradenitis": "inflammatory",
    "Lichenified eczematous dermatitis": "inflammatory", "Traumatic petechiae": "vascular_purpuric",
    "Lichen spinulosus": "inflammatory",
}

def map_category(label):
    if label in CATEGORY_MAP:
        return CATEGORY_MAP[label]
    # keyword fallback for the long tail
    l = label.lower()
    if any(k in l for k in ["dermatitis","eczema","psoriasis","lichen","urticaria","prurig","rosacea","acne"]):
        return "inflammatory"
    if any(k in l for k in ["tinea","herpes","fungal","candida","infection","viral","warts","verruca",
                            "molluscum","scabies","cellulitis","impetigo","syphilis","larva","mycobacter"]):
        return "infectious"
    if any(k in l for k in ["carcinoma","melanoma","nevus","keratosis","lymphoma","sarcoma","neoplasm",
                            "metastasis","xanthoma","fibroma"]):
        return "neoplastic"
    if any(k in l for k in ["purpura","vasculitis","petechiae","ecchymos","livedo","hematoma","varicose","stasis ulcer"]):
        return "vascular_purpuric"
    if any(k in l for k in ["pigment","melasma","vitiligo","hypomelanosis","hyperpigment","hypopigment"]):
        return "pigmentary"
    return "other"

meta_check = pd.read_csv(f"{BASE}/meta_scin.csv")
meta_check["category"] = meta_check["label"].apply(map_category)
print("Category distribution:")
print(meta_check["category"].value_counts())
print(f"\nTotal images: {len(meta_check)}")

Category distribution:
category
inflammatory         3861
infectious           1680
vascular_purpuric     337
other                 257
traumatic_other       169
neoplastic            161
pigmentary             52
Name: count, dtype: int64

Total images: 6517


In [7]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit

BASE = "/kaggle/input/notebooks/nirajankunwor/feature-extraction/embeddings"
MODELS = ["resnet_baseline", "dermlip", "monet", "dinov3"]
rng = np.random.default_rng(42)

CATEGORY_MAP = {
    # --- INFLAMMATORY / ECZEMATOUS / REACTIVE ---
    "Eczema": "inflammatory", "Allergic Contact Dermatitis": "inflammatory",
    "Urticaria": "inflammatory", "Psoriasis": "inflammatory",
    "Drug Rash": "inflammatory", "Acne": "inflammatory",
    "CD - Contact dermatitis": "inflammatory", "Acute dermatitis, NOS": "inflammatory",
    "Pityriasis rosea": "inflammatory", "Keratosis pilaris": "inflammatory",
    "Irritant Contact Dermatitis": "inflammatory", "Lichen Simplex Chronicus": "inflammatory",
    "Lichen planus/lichenoid eruption": "inflammatory", "Stasis Dermatitis": "inflammatory",
    "Rosacea": "inflammatory", "Prurigo nodularis": "inflammatory",
    "Hypersensitivity": "inflammatory", "Photodermatitis": "inflammatory",
    "Acute and chronic dermatitis": "inflammatory", "Intertrigo": "inflammatory",
    "Perioral Dermatitis": "inflammatory", "Seborrheic Dermatitis": "inflammatory",
    "Chronic dermatitis, NOS": "inflammatory", "Miliaria": "inflammatory",
    "Granuloma annulare": "inflammatory", "Insect Bite": "inflammatory",  # MOVED: reactive, not infection
    "Erythema multiforme": "inflammatory", "Cutaneous lupus": "inflammatory",
    "Pityriasis lichenoides": "inflammatory", "Lichen nitidus": "inflammatory",
    "Cutaneous sarcoidosis": "inflammatory", "Hidradenitis": "inflammatory",
    "Lichenified eczematous dermatitis": "inflammatory", "Lichen spinulosus": "inflammatory",
    # --- INFECTIOUS ---
    "Folliculitis": "infectious", "Tinea": "infectious",
    "Impetigo": "infectious", "Herpes Zoster": "infectious", "Herpes Simplex": "infectious",
    "Tinea Versicolor": "infectious", "Viral Exanthem": "infectious",
    "Verruca vulgaris": "infectious", "Scabies": "infectious", "Cellulitis": "infectious",
    "Molluscum Contagiosum": "infectious", "Ecthyma": "infectious",
    "Onychomycosis": "infectious", "Infected eczema": "infectious",
    # --- VASCULAR / PURPURIC ---
    "Pigmented purpuric eruption": "vascular_purpuric", "Leukocytoclastic Vasculitis": "vascular_purpuric",
    "Purpura": "vascular_purpuric", "O/E - ecchymoses present": "vascular_purpuric",
    "Livedo reticularis": "vascular_purpuric", "Traumatic petechiae": "vascular_purpuric",
    # --- NEOPLASTIC / PIGMENTED LESIONS ---
    "Actinic Keratosis": "neoplastic", "SCC/SCCIS": "neoplastic",
    "Hemangioma": "neoplastic", "Dermatofibroma": "neoplastic", "SK/ISK": "neoplastic",
    "Cutaneous T Cell Lymphoma": "neoplastic", "Melanocytic Nevus": "neoplastic",
    "Basal Cell Carcinoma": "neoplastic",
    # --- TRAUMATIC / OTHER ---
    "Abrasion, scrape, or scab": "traumatic_other", "Scar Condition": "traumatic_other",
    "Abscess": "traumatic_other", "Inflicted skin lesions": "traumatic_other",
    "Xerosis": "traumatic_other", "Superficial wound of body region": "traumatic_other",
    # --- PIGMENTARY ---
    "Post-Inflammatory hyperpigmentation": "pigmentary", "Erythema ab igne": "pigmentary",
}

def map_category(label):
    if label in CATEGORY_MAP:
        return CATEGORY_MAP[label]
    l = label.lower()
    if any(k in l for k in ["dermatitis","eczema","psoriasis","lichen","urticaria","prurig","rosacea","acne","bite"]):
        return "inflammatory"
    if any(k in l for k in ["tinea","herpes","fungal","candida","infection","viral","warts","verruca",
                            "molluscum","scabies","cellulitis","impetigo","syphilis","larva","mycobacter"]):
        return "infectious"
    if any(k in l for k in ["carcinoma","melanoma","nevus","keratosis","lymphoma","sarcoma","neoplasm",
                            "metastasis","xanthoma","fibroma"]):
        return "neoplastic"
    if any(k in l for k in ["purpura","vasculitis","petechiae","ecchymos","livedo","hematoma","varicose","stasis ulcer"]):
        return "vascular_purpuric"
    if any(k in l for k in ["pigment","melasma","vitiligo","hypomelanosis","hyperpigment","hypopigment"]):
        return "pigmentary"
    return "other"

def fit_probe(emb, y, groups, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
    tr, te = next(gss.split(emb, y, groups))
    scaler = StandardScaler().fit(emb[tr])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(scaler.transform(emb[tr]), y[tr])
    return te, y[te], clf.predict(scaler.transform(emb[te]))

def boot_ci(y_true, y_pred, metric_fn, n=1000):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    stats = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        try: stats.append(metric_fn(y_true[idx], y_pred[idx]))
        except Exception: pass
    return np.mean(stats), np.percentile(stats, 2.5), np.percentile(stats, 97.5)

# category distribution
meta_check = pd.read_csv(f"{BASE}/meta_scin.csv")
meta_check["category"] = meta_check["label"].apply(map_category)
print("Category distribution (Insect Bite moved to inflammatory):")
print(meta_check["category"].value_counts())

print("\n" + "="*60)
print("SCIN DISTRIBUTION EFFECT — clinical-category probe")
print("balanced accuracy [95% CI]")
print("-"*60)
grouped_results = {}
for m in MODELS:
    emb = np.load(f"{BASE}/{m}_scin.npy")
    meta = pd.read_csv(f"{BASE}/meta_scin.csv"); meta["category"] = meta["label"].apply(map_category)
    y = meta["category"].values; groups = meta["patient_id"].astype(str).values
    te, y_te, pred = fit_probe(emb, y, groups)
    mean, lo, hi = boot_ci(y_te, pred, balanced_accuracy_score)
    grouped_results[m] = (mean, lo, hi)
    print(f"{m:<16} {mean:.3f} [{lo:.3f}, {hi:.3f}]")

print("\nPer-category recall — resnet_baseline (where does the cancer model fail?):")
emb = np.load(f"{BASE}/resnet_baseline_scin.npy")
meta = pd.read_csv(f"{BASE}/meta_scin.csv"); meta["category"] = meta["label"].apply(map_category)
y = meta["category"].values; groups = meta["patient_id"].astype(str).values
te, y_te, pred = fit_probe(emb, y, groups)
for cat in sorted(set(y_te)):
    mask = y_te == cat
    if mask.sum() > 0:
        print(f"  {cat:<20} n={mask.sum():<4} recall={(pred[mask]==cat).mean():.3f}")

Category distribution (Insect Bite moved to inflammatory):
category
inflammatory         4268
infectious           1279
vascular_purpuric     337
other                 251
traumatic_other       169
neoplastic            161
pigmentary             52
Name: count, dtype: int64

SCIN DISTRIBUTION EFFECT — clinical-category probe
balanced accuracy [95% CI]
------------------------------------------------------------
resnet_baseline  0.172 [0.156, 0.187]
dermlip          0.352 [0.298, 0.416]
monet            0.335 [0.268, 0.397]
dinov3           0.270 [0.244, 0.296]

Per-category recall — resnet_baseline (where does the cancer model fail?):
  infectious           n=383  recall=0.316
  inflammatory         n=1275 recall=0.642
  neoplastic           n=47   recall=0.000
  other                n=79   recall=0.051
  pigmentary           n=7    recall=0.000
  traumatic_other      n=56   recall=0.018
  vascular_purpuric    n=113  recall=0.177


In [8]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

BASE = "/kaggle/input/notebooks/nirajankunwor/feature-extraction/embeddings"
MODELS = ["resnet_baseline", "dermlip", "monet", "dinov3"]
rng = np.random.default_rng(42)

CATEGORY_MAP = {
    "Eczema":"inflammatory","Allergic Contact Dermatitis":"inflammatory","Urticaria":"inflammatory",
    "Psoriasis":"inflammatory","Drug Rash":"inflammatory","Acne":"inflammatory",
    "CD - Contact dermatitis":"inflammatory","Acute dermatitis, NOS":"inflammatory",
    "Pityriasis rosea":"inflammatory","Keratosis pilaris":"inflammatory","Irritant Contact Dermatitis":"inflammatory",
    "Lichen Simplex Chronicus":"inflammatory","Lichen planus/lichenoid eruption":"inflammatory",
    "Stasis Dermatitis":"inflammatory","Rosacea":"inflammatory","Prurigo nodularis":"inflammatory",
    "Hypersensitivity":"inflammatory","Photodermatitis":"inflammatory","Acute and chronic dermatitis":"inflammatory",
    "Intertrigo":"inflammatory","Perioral Dermatitis":"inflammatory","Seborrheic Dermatitis":"inflammatory",
    "Chronic dermatitis, NOS":"inflammatory","Miliaria":"inflammatory","Granuloma annulare":"inflammatory",
    "Insect Bite":"inflammatory","Erythema multiforme":"inflammatory","Cutaneous lupus":"inflammatory",
    "Pityriasis lichenoides":"inflammatory","Lichen nitidus":"inflammatory","Cutaneous sarcoidosis":"inflammatory",
    "Hidradenitis":"inflammatory","Lichenified eczematous dermatitis":"inflammatory","Lichen spinulosus":"inflammatory",
    "Folliculitis":"infectious","Tinea":"infectious","Impetigo":"infectious","Herpes Zoster":"infectious",
    "Herpes Simplex":"infectious","Tinea Versicolor":"infectious","Viral Exanthem":"infectious",
    "Verruca vulgaris":"infectious","Scabies":"infectious","Cellulitis":"infectious",
    "Molluscum Contagiosum":"infectious","Ecthyma":"infectious","Onychomycosis":"infectious","Infected eczema":"infectious",
    "Pigmented purpuric eruption":"vascular_purpuric","Leukocytoclastic Vasculitis":"vascular_purpuric",
    "Purpura":"vascular_purpuric","O/E - ecchymoses present":"vascular_purpuric","Livedo reticularis":"vascular_purpuric",
    "Traumatic petechiae":"vascular_purpuric","Actinic Keratosis":"neoplastic","SCC/SCCIS":"neoplastic",
    "Hemangioma":"neoplastic","Dermatofibroma":"neoplastic","SK/ISK":"neoplastic",
    "Cutaneous T Cell Lymphoma":"neoplastic","Melanocytic Nevus":"neoplastic","Basal Cell Carcinoma":"neoplastic",
}
def map_category(label):
    if label in CATEGORY_MAP: return CATEGORY_MAP[label]
    l = label.lower()
    if any(k in l for k in ["dermatitis","eczema","psoriasis","lichen","urticaria","prurig","rosacea","acne","bite"]): return "inflammatory"
    if any(k in l for k in ["tinea","herpes","fungal","candida","infection","viral","verruca","molluscum","scabies","cellulitis","impetigo","syphilis","larva","mycobacter"]): return "infectious"
    if any(k in l for k in ["carcinoma","melanoma","nevus","keratosis","lymphoma","sarcoma","neoplasm","metastasis","xanthoma","fibroma"]): return "neoplastic"
    if any(k in l for k in ["purpura","vasculitis","petechiae","ecchymos","livedo","hematoma","varicose"]): return "vascular_purpuric"
    if any(k in l for k in ["pigment","melasma","vitiligo","hypomelanosis"]): return "pigmentary"
    return "other"

def knn_purity_ci(emb, labels, k=10, n_boot=500):
    emb_s = StandardScaler().fit_transform(emb)
    nn = NearestNeighbors(n_neighbors=k+1, metric="cosine").fit(emb_s)
    _, idx = nn.kneighbors(emb_s); idx = idx[:,1:]
    labels = np.asarray(labels)
    per_point = (labels[idx] == labels[:,None]).mean(axis=1)
    boots = [per_point[rng.integers(0,len(per_point),len(per_point))].mean() for _ in range(n_boot)]
    return per_point.mean(), np.percentile(boots,2.5), np.percentile(boots,97.5)

def run_purity(ds, label_col, min_count):
    print(f"\n{'='*64}\n{ds.upper()} — label='{label_col}', min_count={min_count}\n{'='*64}")
    meta = pd.read_csv(f"{BASE}/meta_{ds}.csv")
    if ds == "scin":
        meta["category"] = meta["label"].apply(map_category)
    vc = meta[label_col].value_counts()
    keep = vc[vc >= min_count].index
    mask = meta[label_col].isin(keep).values
    labels = meta[label_col].values[mask]
    p = pd.Series(labels).value_counts(normalize=True).values
    chance = (p**2).sum()
    print(f"({len(keep)} classes, {mask.sum()} images, chance floor ≈ {chance:.3f})")
    for m in MODELS:
        emb = np.load(f"{BASE}/{m}_{ds}.npy")[mask]
        mean, lo, hi = knn_purity_ci(emb, labels)
        print(f"  {m:<16} purity={mean:.3f} [{lo:.3f}, {hi:.3f}]   lift={mean-chance:+.3f}")

print("ADDITION A: REPRESENTATION QUALITY — label-free kNN neighbor purity")
print("Do frozen features cluster same-condition images without any classifier?")

# FINE-GRAINED (main result)
run_purity("scin", "label", min_count=20)
run_purity("ddi", "label", min_count=15)
run_purity("hamisic_test", "mapped_label", min_count=20)

# CATEGORICAL (secondary)
run_purity("scin", "category", min_count=50)

ADDITION A: REPRESENTATION QUALITY — label-free kNN neighbor purity
Do frozen features cluster same-condition images without any classifier?

SCIN — label='label', min_count=20
(47 classes, 5713 images, chance floor ≈ 0.069)
  resnet_baseline  purity=0.131 [0.127, 0.134]   lift=+0.062
  dermlip          purity=0.297 [0.290, 0.303]   lift=+0.228
  monet            purity=0.211 [0.207, 0.215]   lift=+0.142
  dinov3           purity=0.228 [0.224, 0.233]   lift=+0.159

DDI — label='label', min_count=15
(11 classes, 437 images, chance floor ≈ 0.137)
  resnet_baseline  purity=0.207 [0.190, 0.227]   lift=+0.070
  dermlip          purity=0.295 [0.274, 0.318]   lift=+0.158
  monet            purity=0.255 [0.238, 0.274]   lift=+0.118
  dinov3           purity=0.253 [0.232, 0.273]   lift=+0.116

HAMISIC_TEST — label='mapped_label', min_count=20
(8 classes, 5378 images, chance floor ≈ 0.365)
  resnet_baseline  purity=0.783 [0.776, 0.791]   lift=+0.419
  dermlip          purity=0.709 [0.700, 0.717]

In [9]:
import json
addition_a = {
    "knn_purity_scin_fine": {"resnet_baseline":[0.131,0.062],"dermlip":[0.297,0.228],"monet":[0.211,0.142],"dinov3":[0.228,0.159]},
    "knn_purity_indomain": {"resnet_baseline":[0.783,0.419],"dermlip":[0.709,0.344],"monet":[0.657,0.293],"dinov3":[0.657,0.292]},
    "knn_purity_scin_category": {"resnet_baseline":[0.524,0.043],"dermlip":[0.629,0.148],"monet":[0.576,0.095],"dinov3":[0.579,0.098]},
    "note": "purity=[mean, lift_over_chance]"
}
with open("/kaggle/working/addition_a_results.json","w") as f:
    json.dump(addition_a, f, indent=2)
print("Saved Addition A results")

Saved Addition A results


In [10]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit

BASE = "/kaggle/input/notebooks/nirajankunwor/feature-extraction/embeddings"
MODELS = ["resnet_baseline", "dermlip", "monet", "dinov3"]
rng = np.random.default_rng(42)

# SCIN category mapping (reuse)
# (assumes map_category is still defined from Addition A; if session reset, re-run that block first)

def patient_split(meta, group_col, seed=42, test_size=0.3):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    return next(gss.split(np.arange(len(meta)), groups=meta[group_col].astype(str).values))

def bacc(clf, Xte, yte):
    return balanced_accuracy_score(yte, clf.predict(Xte))

print("ADDITION B: LOW-COMPUTE ADAPTATION FIXES on SCIN (clinical categories)")
print("Does recovery track the latent structure from Addition A?\n")

results = {}
for m in MODELS:
    emb = np.load(f"{BASE}/{m}_scin.npy")
    meta = pd.read_csv(f"{BASE}/meta_scin.csv")
    meta["category"] = meta["label"].apply(map_category)
    y = meta["category"].values
    tr, te = patient_split(meta, "patient_id")
    ytr, yte = y[tr], y[te]

    # --- Fix 0: full probe (reference ceiling) ---
    sc = StandardScaler().fit(emb[tr])
    Xtr, Xte = sc.transform(emb[tr]), sc.transform(emb[te])
    full = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, ytr)
    full_bacc = bacc(full, Xte, yte)

    # --- Fix 1: few-shot (k=10 per class) ---
    fewshot_baccs = []
    for seed in range(5):  # average over 5 few-shot draws
        r = np.random.default_rng(seed)
        idx = []
        for c in np.unique(ytr):
            ci = np.where(ytr == c)[0]
            idx.extend(r.choice(ci, min(10, len(ci)), replace=False))
        idx = np.array(idx)
        fs = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr[idx], ytr[idx])
        fewshot_baccs.append(bacc(fs, Xte, yte))
    fewshot_bacc = np.mean(fewshot_baccs)

    # --- Fix 2: feature standardization already applied above; compare to NO standardization ---
    raw = LogisticRegression(max_iter=2000, class_weight="balanced").fit(emb[tr], ytr)
    raw_bacc = bacc(raw, emb[te], yte)
    std_gain = full_bacc - raw_bacc  # benefit of standardization

    results[m] = {"raw": raw_bacc, "standardized": full_bacc,
                  "fewshot_k10": fewshot_bacc, "std_gain": std_gain}
    print(f"{m:<16} raw={raw_bacc:.3f}  +std={full_bacc:.3f}  fewshot(k=10)={fewshot_bacc:.3f}")

# --- The A+B link: does full-probe performance track latent purity? ---
print("\n" + "="*64)
print("A+B LINK: latent structure (Addition A) vs recoverable performance (B)")
print("="*64)
purity_scin = {"resnet_baseline":0.062, "dermlip":0.228, "monet":0.142, "dinov3":0.159}  # lift from A
print(f"{'Model':<16}{'A: purity_lift':<16}{'B: full_bacc':<14}{'B: fewshot':<12}")
for m in MODELS:
    print(f"{m:<16}{purity_scin[m]:<16.3f}{results[m]['standardized']:<14.3f}{results[m]['fewshot_k10']:<12.3f}")

# correlation between latent structure and recoverability
import numpy as np
a_vals = [purity_scin[m] for m in MODELS]
b_vals = [results[m]['standardized'] for m in MODELS]
corr = np.corrcoef(a_vals, b_vals)[0,1]
print(f"\nCorrelation (latent purity vs full-probe accuracy): r = {corr:.3f}")

ADDITION B: LOW-COMPUTE ADAPTATION FIXES on SCIN (clinical categories)
Does recovery track the latent structure from Addition A?

resnet_baseline  raw=0.214  +std=0.207  fewshot(k=10)=0.194
dermlip          raw=0.421  +std=0.359  fewshot(k=10)=0.411
monet            raw=0.332  +std=0.342  fewshot(k=10)=0.389
dinov3           raw=0.352  +std=0.311  fewshot(k=10)=0.324

A+B LINK: latent structure (Addition A) vs recoverable performance (B)
Model           A: purity_lift  B: full_bacc  B: fewshot  
resnet_baseline 0.062           0.207         0.194       
dermlip         0.228           0.359         0.411       
monet           0.142           0.342         0.389       
dinov3          0.159           0.311         0.324       

Correlation (latent purity vs full-probe accuracy): r = 0.903


In [11]:
import json
addition_b = {
    "fixes_scin_category": {
        m: results[m] for m in MODELS
    },
    "ab_link_correlation_r": 0.903,
    "purity_lift_A": purity_scin,
    "caveats": "n=4 models (correlation suggestive not significant); standardization did not help; few-shot approx equals full probe"
}
with open("/kaggle/working/addition_b_results.json","w") as f:
    json.dump(addition_b, f, indent=2)
print("Saved Addition B results")

Saved Addition B results


In [12]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, json, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors

# ---- check the saved JSON result files if they exist ----
print("="*60); print("SAVED RESULT FILES"); print("="*60)
for f in ["core_results.json","addition_a_results.json","addition_b_results.json"]:
    for path in [f"/kaggle/working/{f}", f]:
        if os.path.exists(path):
            print(f"\n--- {path} ---")
            print(json.dumps(json.load(open(path)), indent=2))
            break
    else:
        print(f"\n--- {f}: NOT FOUND (will recompute below) ---")

SAVED RESULT FILES

--- core_results.json: NOT FOUND (will recompute below) ---

--- /kaggle/working/addition_a_results.json ---
{
  "knn_purity_scin_fine": {
    "resnet_baseline": [
      0.131,
      0.062
    ],
    "dermlip": [
      0.297,
      0.228
    ],
    "monet": [
      0.211,
      0.142
    ],
    "dinov3": [
      0.228,
      0.159
    ]
  },
  "knn_purity_indomain": {
    "resnet_baseline": [
      0.783,
      0.419
    ],
    "dermlip": [
      0.709,
      0.344
    ],
    "monet": [
      0.657,
      0.293
    ],
    "dinov3": [
      0.657,
      0.292
    ]
  },
  "knn_purity_scin_category": {
    "resnet_baseline": [
      0.524,
      0.043
    ],
    "dermlip": [
      0.629,
      0.148
    ],
    "monet": [
      0.576,
      0.095
    ],
    "dinov3": [
      0.579,
      0.098
    ]
  },
  "note": "purity=[mean, lift_over_chance]"
}

--- /kaggle/working/addition_b_results.json ---
{
  "fixes_scin_category": {
    "resnet_baseline": {
      "raw": 0.214